# ResNet-18 Robust

This notebook trains a pretrained ResNet-18 on a robust dataset which combrises of a mix of SID and CIFAKE.

- Label `0`: real
- Label `1`: AI-generated/fake
- Input size: 224 x 224
- Model selection: highest validation F1

## Check that GPU exists

In [44]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available")

CUDA available: True
GPU: Tesla T4


## Set up project

In [43]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = Path("/content/ai-image-detector")

if not PROJECT_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/mikkichan22/AI-image-detector.git",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

(PROJECT_ROOT / "src" / "__init__.py").touch()

print("Working directory:", Path.cwd())
print("Project exists:", PROJECT_ROOT.exists())
print("src exists:", (PROJECT_ROOT / "src").exists())

Working directory: /content/ai-image-detector
Project exists: True
src exists: True


In [5]:
!pip install -q datasets

## Create sid subset

In [11]:
!python -m src.create_sid_subset

Loading SID_Set in streaming mode...
README.md: 100% 3.30k/3.30k [00:00<00:00, 7.76MB/s]
Resolving data files: 100% 249/249 [00:00<00:00, 447081.21it/s]
Resolving data files: 100% 34/34 [00:00<00:00, 78701.07it/s]
Resolving data files: 100% 249/249 [00:00<00:00, 417085.34it/s]
Resolving data files: 100% 34/34 [00:00<00:00, 264085.81it/s]
Saved: {1: 504, 0: 496}
Saved: {1: 995, 0: 1005}
Saved: {1: 1487, 0: 1513}
Saved: {1: 1967, 0: 2033}
Saved: {1: 2472, 0: 2528}
Saved: {1: 2982, 0: 3018}
Saved: {1: 3494, 0: 3506}
Saved: {1: 4007, 0: 3993}
Saved: {1: 4526, 0: 4474}
Saved: {1: 5029, 0: 4971}
Saved: {1: 5543, 0: 5457}
Saved: {1: 6033, 0: 5967}
Saved: {1: 6537, 0: 6463}
Saved: {1: 7033, 0: 6967}
Saved: {1: 7556, 0: 7444}
Saved: {1: 8019, 0: 7981}
Saved: {1: 8513, 0: 8487}
Saved: {1: 9009, 0: 8991}
Saved: {1: 9527, 0: 9473}
Saved: {1: 10000, 0: 10000}

Finished.
Images saved: {1: 10000, 0: 10000}
Output: /content/ai-image-detector/data/raw/SID_Set_subset


## Check that it exists

In [45]:
!find data/raw/SID_Set_subset -type f | wc -l
!du -sh data/raw/SID_Set_subset

20000
4.5G	data/raw/SID_Set_subset


## Create SID splits

In [62]:
!python -m src.create_sid_splits

Images found: 20000
Duplicate groups: 2
Created: /content/ai-image-detector/data/sid_splits.csv
Total images: 20000

Final split counts:
train      label=0: 7000
train      label=1: 7002
validation label=0: 1500
validation label=1: 1499
test       label=0: 1500
test       label=1: 1499


In [63]:
!find src -maxdepth 1 -type f | sort

src/combine_splits.py
src/create_robustness_sets.py
src/create_sid_splits.py
src/create_sid_subset.py
src/create_splits.py
src/data.py
src/evaluate.py
src/__init__.py
src/inspect_dataset.py
src/train_resnet.py
src/transforms.py


# Download CIFAKE dataset

## **1. Upload kaggle.json file**
Go to kaggle account -> settings -> API tokens -> create legacy API key

In [52]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


## **2. Configure kaggle**

In [53]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

## **3. Download and Extract CIFAKE dataset**

### Download

In [54]:
!mkdir -p /content/cifake_download
!kaggle datasets download \
    -d birdy654/cifake-real-and-ai-generated-synthetic-images \
    -p /content/cifake_download

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other
100% 105M/105M [00:03<00:00, 32.6MB/s]



### Extract

In [55]:
!mkdir -p /content/ai-image-detector/data/raw
!unzip -q /content/cifake_download/*.zip \
    -d /content/ai-image-detector/data/raw/CIFAKE

### inspect the extracted structure

In [56]:
!find /content/ai-image-detector/data/raw/CIFAKE -maxdepth 4 -type d | sort

/content/ai-image-detector/data/raw/CIFAKE
/content/ai-image-detector/data/raw/CIFAKE/test
/content/ai-image-detector/data/raw/CIFAKE/test/FAKE
/content/ai-image-detector/data/raw/CIFAKE/test/REAL
/content/ai-image-detector/data/raw/CIFAKE/train
/content/ai-image-detector/data/raw/CIFAKE/train/FAKE
/content/ai-image-detector/data/raw/CIFAKE/train/REAL


## Create combined splits (SID and CIFAKE)

In [64]:
!python -m src.combine_splits

/content/ai-image-detector/src/combine_splits.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
Created: /content/ai-image-detector/data/combined_splits.csv
source_dataset  split       label
CIFAKE          test        0        10000
                            1        10378
                train       0         2500
                            1         2500
                validation  0         7500
                            1         7456
SID_Set         test        0         1500
                            1         1499
                train       0         7000
                            1         7002
                validation  0         1500
                            1 

## Rewrite combined splits so validation set is mostly SID with some CIFAKE (60/40 ratio)

In [66]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/ai-image-detector")

combined_path = (
    PROJECT_ROOT / "data" / "combined_splits.csv"
)

priority_path = (
    PROJECT_ROOT / "data" / "sid_priority_splits.csv"
)

combined = pd.read_csv(combined_path)

# Keep every training and test row.
non_validation = combined[
    combined["split"] != "validation"
].copy()

# Keep all SID validation rows.
sid_validation = combined[
    (combined["split"] == "validation")
    & (combined["source_dataset"] == "SID_Set")
].copy()

# Get CIFAKE validation rows.
cifake_validation = combined[
    (combined["split"] == "validation")
    & (combined["source_dataset"] == "CIFAKE")
].copy()

# Select a balanced CIFAKE validation sample.
# This selects 1,000 REAL and 1,000 FAKE images.
cifake_validation_sample = pd.concat(
    [
        group.sample(
            n=min(len(group), 1000),
            random_state=42,
        )
        for _, group in cifake_validation.groupby("label")
    ],
    ignore_index=True,
)

# Construct the new manifest.
sid_priority = pd.concat(
    [
        non_validation,
        sid_validation,
        cifake_validation_sample,
    ],
    ignore_index=True,
)

sid_priority.to_csv(
    priority_path,
    index=False,
)

print("Created:", priority_path)

Created: /content/ai-image-detector/data/sid_priority_splits.csv


## inspect result

In [67]:
import pandas as pd

combined = pd.read_csv("data/sid_priority_splits.csv")

display(
    combined.groupby(
        ["source_dataset", "split", "label"]
    ).size()
)

source_dataset  split       label
CIFAKE          test        0        10000
                            1        10378
                train       0         2500
                            1         2500
                validation  0         1000
                            1         1000
SID_Set         test        0         1500
                            1         1499
                train       0         7000
                            1         7002
                validation  0         1500
                            1         1499
dtype: int64

## check if there's missing paths

In [69]:
missing = []

for path_string in sid_priority["image_path"]:
    path = Path(path_string)

    if not path.is_absolute():
        path = PROJECT_ROOT / path

    if not path.exists():
        missing.append(str(path))

print("Missing paths:", len(missing))

Missing paths: 0


# Train one epoch as a test

In [59]:
!python -m src.train_resnet \
    --project_root . \
    --splits_file data/combined_splits.csv \
    --output_dir /content/drive/MyDrive/ai-image-detector/results/sid_mixed_test \
    --checkpoint_dir /content/drive/MyDrive/ai-image-detector/checkpoints/sid_mixed_test \
    --epochs 1 \
    --freeze_epochs 1 \
    --batch_size 32 \
    --num_workers 2 \
    --seed 42

Using device: cuda
Reading split file: data/combined_splits.csv
Training images: 22001
Validation images: 17955
Test images: 20378
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 201MB/s]

Epoch 1/1
Train loss: 0.5806, Train F1: 0.6986
Validation loss: 0.5549, Validation F1: 0.7598
Saved: /content/drive/MyDrive/ai-image-detector/checkpoints/sid_mixed_test/resnet18_clean_best.pth
                                     
Final test results:
Accuracy:  0.7093
Precision: 0.6600
Recall:    0.8855
F1:        0.7563

Saved:
/content/drive/MyDrive/ai-image-detector/results/sid_mixed_test/training_history.csv
/content/drive/MyDrive/ai-image-detector/results/sid_mixed_test/test_metrics.json


# Train final model

In [ ]:
!python -m src.train_resnet \
    --project_root . \
    --splits_file data/sid_priority_splits.csv \
    --output_dir /content/drive/MyDrive/ai-image-detector/results/sid_priority_augmented \
    --checkpoint_dir /content/drive/MyDrive/ai-image-detector/checkpoints/sid_priority_augmented \
    --epochs 10 \
    --freeze_epochs 2 \
    --batch_size 64 \
    --num_workers 2 \
    --seed 42

Using device: cuda
Reading split file: data/sid_priority_splits.csv
Training images: 19002
Validation images: 4999
Test images: 23377

Epoch 1/10
Train loss: 0.6252, Train F1: 0.6590
Validation loss: 0.5581, Validation F1: 0.7581
Saved: /content/drive/MyDrive/ai-image-detector/checkpoints/sid_priority_augmented/resnet18_clean_best.pth

Epoch 2/10
Train loss: 0.5128, Train F1: 0.7681
Validation loss: 0.4831, Validation F1: 0.8101
Saved: /content/drive/MyDrive/ai-image-detector/checkpoints/sid_priority_augmented/resnet18_clean_best.pth

Epoch 3/10
Unfreezing full ResNet-18
 54% 161/297 [02:24<02:02,  1.11it/s]